# Data Source Exploration
Explore each raw data source from the marketing pipeline.

This notebook initialises each extractor with a small, fast date range and
examines schema, distributions, and initial data-quality observations.

In [ ]:
import sys
from pathlib import Path

# Allow imports from the project root (notebooks run from notebooks/)
sys.path.insert(0, str(Path('.').parent))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:,.2f}'.format)

from datetime import datetime, timedelta
from config import BASE_DIR

print(f"Project root: {BASE_DIR}")
print(f"pandas {pd.__version__} | numpy {np.__version__}")


In [ ]:
from src.extractors import (
    CRMExtractor,
    MetaAdsExtractor,
    GoogleAdsExtractor,
    GA4Extractor,
    EmailPlatformExtractor,
)

# Use a small date range to keep extraction fast in the notebook
end_date   = datetime(2025, 3, 1)
start_date = datetime(2025, 1, 1)

print(f"Extracting data from {start_date.date()} to {end_date.date()}...")

# Lightweight configs: fewer records so cells run quickly
crm_ext   = CRMExtractor({"accounts_count": 200, "contacts_count": 600,
                           "opportunities_count": 120, "activities_count": 800})
meta_ext  = MetaAdsExtractor({"campaigns": 5})
gads_ext  = GoogleAdsExtractor({"campaigns": 4})
ga4_ext   = GA4Extractor({"sessions_count": 2000, "events_count": 8000})
email_ext = EmailPlatformExtractor({"campaigns": 8, "sends_count": 2000})

crm_raw   = crm_ext.extract(start_date, end_date)
meta_raw  = meta_ext.extract(start_date, end_date)
gads_raw  = gads_ext.extract(start_date, end_date)
ga4_raw   = ga4_ext.extract(start_date, end_date)
email_raw = email_ext.extract(start_date, end_date)

print("Extraction complete.")
print(f"  CRM records       : {len(crm_raw):,}")
print(f"  Meta Ads records  : {len(meta_raw):,}")
print(f"  Google Ads records: {len(gads_raw):,}")
print(f"  GA4 records       : {len(ga4_raw):,}")
print(f"  Email records     : {len(email_raw):,}")


In [ ]:
# ── CRM data ──────────────────────────────────────────────────────────────
print("=== CRM – Accounts (sample) ===")
accounts = crm_raw[crm_raw['record_type'] == 'account'].copy()
print(f"Shape: {accounts.shape}")
display(accounts.head())
print()
print("Column dtypes:")
print(accounts.dtypes)
print()
print("Numeric summary:")
display(accounts.describe())


In [ ]:
# ── Meta Ads data ─────────────────────────────────────────────────────────
print("=== Meta Ads ===")
print(f"Shape: {meta_raw.shape}")
print(f"Date range: {meta_raw['date'].min()}  →  {meta_raw['date'].max()}")
print(f"Column dtypes:")
print(meta_raw.dtypes)
print()
display(meta_raw.head())


In [ ]:
# ── Google Ads data ───────────────────────────────────────────────────────
print("=== Google Ads ===")
print(f"Shape: {gads_raw.shape}")
print(f"Date range: {gads_raw['date'].min()}  →  {gads_raw['date'].max()}")
display(gads_raw.head())


In [ ]:
# ── GA4 session data ──────────────────────────────────────────────────────
print("=== GA4 Sessions ===")
ga4_sessions = ga4_raw[ga4_raw.get('record_type', pd.Series(['session']*len(ga4_raw))) == 'session'].copy()     if 'record_type' in ga4_raw.columns else ga4_raw.copy()
print(f"Shape: {ga4_sessions.shape}")
display(ga4_sessions.head())
print()
numeric_cols = ga4_sessions.select_dtypes(include='number').columns.tolist()
if numeric_cols:
    display(ga4_sessions[numeric_cols].describe())


In [ ]:
# ── Email data ────────────────────────────────────────────────────────────
print("=== Email Engagement ===")
print(f"Shape: {email_raw.shape}")
display(email_raw.head())
print()
numeric_cols = email_raw.select_dtypes(include='number').columns.tolist()
if numeric_cols:
    display(email_raw[numeric_cols].describe())


In [ ]:
# ── Data volume summary table ─────────────────────────────────────────────
summary = pd.DataFrame([
    {"Source": "CRM (all records)",   "Rows": len(crm_raw),   "Columns": crm_raw.shape[1]},
    {"Source": "CRM Accounts",        "Rows": len(accounts),  "Columns": accounts.shape[1]},
    {"Source": "Meta Ads",            "Rows": len(meta_raw),  "Columns": meta_raw.shape[1]},
    {"Source": "Google Ads",          "Rows": len(gads_raw),  "Columns": gads_raw.shape[1]},
    {"Source": "GA4",                 "Rows": len(ga4_raw),   "Columns": ga4_raw.shape[1]},
    {"Source": "Email",               "Rows": len(email_raw), "Columns": email_raw.shape[1]},
])
print("=== Data Volume Summary ===")
display(summary.set_index("Source"))


In [ ]:
# ── Distribution plots ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("Key Metric Distributions Across Sources", fontsize=14, fontweight='bold')

# Meta Ads – daily spend
axes[0, 0].hist(meta_raw['spend'].dropna(), bins=30, color='#1877F2', edgecolor='white', linewidth=0.5)
axes[0, 0].set_title("Meta Ads: Daily Spend ($)")
axes[0, 0].set_xlabel("Spend")
axes[0, 0].set_ylabel("Frequency")

# Google Ads – daily spend (field is 'cost')
spend_col = 'cost' if 'cost' in gads_raw.columns else 'spend'
axes[0, 1].hist(gads_raw[spend_col].dropna(), bins=30, color='#4285F4', edgecolor='white', linewidth=0.5)
axes[0, 1].set_title("Google Ads: Daily Spend ($)")
axes[0, 1].set_xlabel("Spend")

# Meta CTR
if 'ctr' in meta_raw.columns:
    axes[0, 2].hist(meta_raw['ctr'].dropna() * 100, bins=30, color='#E1306C', edgecolor='white', linewidth=0.5)
    axes[0, 2].set_title("Meta Ads: CTR (%)")
    axes[0, 2].set_xlabel("CTR (%)")

# GA4 session duration
if 'session_duration_seconds' in ga4_sessions.columns:
    axes[1, 0].hist(ga4_sessions['session_duration_seconds'].clip(0, 600).dropna(),
                    bins=40, color='#34A853', edgecolor='white', linewidth=0.5)
    axes[1, 0].set_title("GA4: Session Duration (s, clipped 600s)")
    axes[1, 0].set_xlabel("Seconds")

# Email open rate by campaign (bar chart of top campaigns)
if 'campaign_id' in email_raw.columns and 'opened' in email_raw.columns and 'delivered' in email_raw.columns:
    camp_stats = (
        email_raw.groupby('campaign_id')
        .agg(delivered=('delivered', 'sum'), opened=('opened', 'sum'))
        .assign(open_rate=lambda d: d['opened'] / d['delivered'].clip(lower=1))
        .sort_values('open_rate', ascending=False)
        .head(8)
    )
    axes[1, 1].barh(camp_stats.index.astype(str), camp_stats['open_rate'] * 100, color='#FF9500')
    axes[1, 1].set_title("Email: Open Rate by Campaign (top 8)")
    axes[1, 1].set_xlabel("Open Rate (%)")
    axes[1, 1].invert_yaxis()

# CRM lead status distribution
if 'lead_status' in crm_raw.columns:
    status_counts = crm_raw['lead_status'].dropna().value_counts()
    axes[1, 2].bar(status_counts.index, status_counts.values, color='#7B2FBE', edgecolor='white')
    axes[1, 2].set_title("CRM: Lead Status Distribution")
    axes[1, 2].set_xlabel("Status")
    axes[1, 2].set_ylabel("Count")
    axes[1, 2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(BASE_DIR / "visuals" / "nb01_distributions.png", dpi=100, bbox_inches='tight')
plt.show()
print("Plot saved to visuals/nb01_distributions.png")


In [ ]:
# ── Initial data-quality observations ─────────────────────────────────────
def null_profile(df, name):
    total = len(df)
    null_pct = (df.isnull().sum() / total * 100).rename("null_pct_%")
    result = null_pct[null_pct > 0].sort_values(ascending=False)
    if result.empty:
        print(f"{name}: No null values detected.")
    else:
        print(f"\n{name} – columns with nulls (% of {total:,} rows):")
        display(result.to_frame())

null_profile(crm_raw,   "CRM")
null_profile(meta_raw,  "Meta Ads")
null_profile(gads_raw,  "Google Ads")
null_profile(ga4_raw,   "GA4")
null_profile(email_raw, "Email")
